# Auto-Experiment Tutorial

This tutorial shows possible ways to utilize auto-experiment functionality
Sections of the following notebook include:
1) Data and model setup
2) Auto-Experiment setup

In [2]:
# Framework
import torch
from torch.utils.data import DataLoader
# Plug and Play XAI Manager
from pnpxai.utils import set_seed
from pnpxai import Project
# Tools
from helpers import get_imagenet_dataset, get_torchvision_model, denormalize_image

## Data and model setup

Initially, we set the seed values, and device.

In [ ]:
set_seed(seed=0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

`helpers` package provides us with the convenient way to load `torchvision` models. Given a model and transformations it requires, we apply those to the dataset to further pass it into the `DataLoader`

In [ ]:
model, transform = get_torchvision_model("resnet18")
model = model.to(device)
dataset = get_imagenet_dataset(transform, subset_size=25)
loader = DataLoader(dataset, batch_size=10)

## Auto-Experiment setup
Having prepared required model and data, we define Project, and Auto-Experiment parameters.

Initially, we initialize the `Project`, which serves as a container for project-specific experiments. We are required to provide name for the project, which is selected to be "Test Project 1". Given the project instance, it is now possible to populate it with experiments.

First is an experiment under "Resnet Experiment" name, it takes `input_extractor`, `target_extractor`, `input_visualizer`, `target_visualizer` parameters.

These functions have default implementations in the `AutoExperiment` class. However, we can customize those to have complete control over the data flow. `input_extractor` and `target_extractor` functions format the batch data supplied to the model.

Likewise, `input_visualizer` and `target_visualizer` allow controlling the data shown on the web ui of the experiment.

In [ ]:
project = Project('Test Project 1')

def input_extractor(x): return x[0].to(device)
def target_extractor(x): return x[1].to(device)

def input_visualizer(x): return denormalize_image(x, transform.mean, transform.std)
def target_visualizer(x): return dataset.dataset.idx_to_label(x.item())

experiment_resnet = project.create_auto_experiment(
    model,
    loader,
    name='Resnet Experiment',
    input_extractor=input_extractor,
    target_extractor=target_extractor,
    input_visualizer=input_visualizer,
    target_visualizer=target_visualizer,
)

Final step in the system communication is the launch of the server of the desired `port` and `host`

In [ ]:
project.get_server().serve(debug=True, host='0.0.0.0', port=5001)